# SIH26158 — KABR DJI_0677 (13-min drone video)

This notebook downloads **DJI_0677.MP4** (3.75 GB, ~13 min) and **DJI_0677.SRT** (9.4 MB)
directly from the HuggingFace dataset
[imageomics/KABR-raw-videos](https://huggingface.co/datasets/imageomics/KABR-raw-videos/tree/main/18_01_2023_session_5),
then runs the complete SIH26158 drone-to-3D pipeline.

### Before you start
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload your `SIH26158/` project folder to `MyDrive/SIH26158_KABR_0677/` (must contain `requirements.txt`, `sih_drone_pipeline/`, and `tests/`).
3. Run cells **in order**. Cell 2 restarts the runtime once — continue from Cell 3 after that.

### Outputs
PLY, LAS, GLB, OBJ, GeoTIFF, run report, and verification report saved to
`MyDrive/SIH26158_KABR_0677/outputs/kabr_0677/`.

In [ ]:
# CELL 1 — verify GPU and mount Google Drive
import os, subprocess
from google.colab import drive

drive.mount('/content/drive')

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
assert gpu.returncode == 0, 'No GPU detected. Select a T4 GPU runtime before continuing.'
print('GPU:', gpu.stdout.strip())

PROJECT_DIR = '/content/drive/MyDrive/SIH26158_KABR_0677'
assert os.path.isfile(f'{PROJECT_DIR}/requirements.txt'), (
    f'Project not found at {PROJECT_DIR}. '
    'Upload your SIH26158 folder to MyDrive/SIH26158_KABR_0677/.'
)
print('Project:', PROJECT_DIR)

In [ ]:
# CELL 2 — install Conda (restarts the runtime once; continue from Cell 3)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 — install CUDA COLMAP and Python dependencies
import os, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SIH26158_KABR_0677'

assert shutil.which('mamba'), 'Conda is missing. Run Cell 2 and wait for its restart.'

pin = '/usr/local/conda-meta/pinned'
if os.path.exists(pin):
    os.remove(pin)

subprocess.run(
    ['mamba', 'install', '-y', '-q', '-c', 'conda-forge',
     'colmap', 'libfaiss', 'openimageio'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     f'{PROJECT_DIR}/requirements.txt'],
    check=True,
)

check = subprocess.run(['colmap', '-h'], capture_output=True, text=True, errors='replace')
header = (check.stdout or '') + (check.stderr or '')
print('\n'.join(header.splitlines()[:10]))
assert check.returncode == 0 and 'COLMAP' in header, 'COLMAP does not run'
assert 'CUDA' in header, 'This COLMAP build does not report CUDA support'

# Run unit tests
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_DIR, check=True,
)
print('Installation and tests complete.')

In [ ]:
# CELL 4 — download DJI_0677.MP4 and DJI_0677.SRT from HuggingFace
import os, time, urllib.request

HF_BASE = (
    'https://huggingface.co/datasets/imageomics/KABR-raw-videos'
    '/resolve/main/18_01_2023_session_5'
)

INPUT_DIR = '/content/kabr_0677_input'
os.makedirs(INPUT_DIR, exist_ok=True)

VIDEO = f'{INPUT_DIR}/DJI_0677.MP4'
TELEMETRY = f'{INPUT_DIR}/DJI_0677.SRT'

files = {
    VIDEO:     f'{HF_BASE}/DJI_0677.MP4',
    TELEMETRY: f'{HF_BASE}/DJI_0677.SRT',
}

for local_path, url in files.items():
    if os.path.isfile(local_path):
        size_mb = os.path.getsize(local_path) / 1024 / 1024
        print(f'Already exists: {os.path.basename(local_path)} ({size_mb:.1f} MB)')
        continue
    print(f'Downloading {os.path.basename(local_path)} from HuggingFace...')
    t0 = time.time()
    urllib.request.urlretrieve(url, local_path)
    elapsed = time.time() - t0
    size_mb = os.path.getsize(local_path) / 1024 / 1024
    print(f'  Done: {size_mb:.1f} MB in {elapsed:.0f}s '
          f'({size_mb/elapsed:.1f} MB/s)')

print(f'\nVideo:     {VIDEO} ({os.path.getsize(VIDEO)/1024/1024:.1f} MB)')
print(f'Telemetry: {TELEMETRY} ({os.path.getsize(TELEMETRY)/1024:.0f} KB)')

In [ ]:
# CELL 5 — configure the run
import os

PROJECT_DIR = '/content/drive/MyDrive/SIH26158_KABR_0677'
INPUT_DIR = '/content/kabr_0677_input'

RUN_NAME = 'kabr_0677'
VIDEO = f'{INPUT_DIR}/DJI_0677.MP4'
TELEMETRY = f'{INPUT_DIR}/DJI_0677.SRT'

#  --- Tunable parameters ---
QUALITY = 'draft'               # 'draft' first; 'full' for final output
TARGET_FRAMES = 250             # ~13 min video at ~1 fps ≈ 780 frames; 250 keeps draft fast
MAX_WIDTH = 1920                # extract resolution
DSM_RESOLUTION_M = 0.5
AI_MASK_DYNAMIC_OBJECTS = True   # YOLO segmentation for animals/people/vehicles
STRICT_CAPTURE_QUALITY = False
RESET_WORKSPACE = True

WORKSPACE = f'/content/sih26158_{RUN_NAME}'     # fast local SSD
OUTPUT = f'{PROJECT_DIR}/outputs/{RUN_NAME}'    # persistent Drive output

assert os.path.isfile(VIDEO), f'Video not found: {VIDEO}'
assert os.path.isfile(TELEMETRY), f'Telemetry not found: {TELEMETRY}'
print('Video:', VIDEO)
print('Telemetry:', TELEMETRY)
print('Output:', OUTPUT)

In [ ]:
# CELL 6 — preflight, GPU monitoring, pipeline run, verification
import os, subprocess, sys, time, shutil

env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')

os.makedirs(OUTPUT, exist_ok=True)
if RESET_WORKSPACE and os.path.isdir(WORKSPACE):
    shutil.rmtree(WORKSPACE)

# Preflight
preflight = [
    sys.executable, '-m', 'sih_drone_pipeline', 'preflight',
    '--video', VIDEO, '--telemetry', TELEMETRY,
    '--workspace', WORKSPACE, '--output', f'{OUTPUT}/preflight.json',
]
assert subprocess.run(preflight, cwd=PROJECT_DIR, env=env).returncode == 0, \
    'Preflight failed; open preflight.json'

# Build command
command = [
    sys.executable, '-m', 'sih_drone_pipeline', 'run',
    '--video', VIDEO, '--telemetry', TELEMETRY,
    '--workspace', WORKSPACE, '--output', OUTPUT,
    '--quality', QUALITY, '--target-frames', str(TARGET_FRAMES),
    '--max-width', str(MAX_WIDTH), '--dsm-resolution', str(DSM_RESOLUTION_M),
    '--keyframe-mode', 'geometry', '--mapper', 'pose-prior',
    '--gps-prior-std-m', '5.0',
]
if AI_MASK_DYNAMIC_OBJECTS:
    command.append('--ai-mask-dynamic')
if STRICT_CAPTURE_QUALITY:
    command.append('--strict-capture-quality')

# GPU monitor
stop_file = '/content/sih_gpu_monitor_0677.stop'
if os.path.exists(stop_file):
    os.remove(stop_file)
monitor = subprocess.Popen([
    sys.executable, '-m', 'sih_drone_pipeline.gpu_monitor',
    '--output', f'{OUTPUT}/gpu_usage.csv', '--stop-file', stop_file,
], cwd=PROJECT_DIR, env=env)

# Run
print('Running:', ' '.join(command))
started = time.time()
try:
    result = subprocess.run(command, cwd=PROJECT_DIR, env=env)
finally:
    open(stop_file, 'w').close()
    monitor.wait(timeout=20)
    if os.path.isdir(f'{WORKSPACE}/logs'):
        shutil.copytree(f'{WORKSPACE}/logs', f'{OUTPUT}/logs', dirs_exist_ok=True)
    partial = f'{WORKSPACE}/run_report.partial.json'
    if os.path.isfile(partial):
        shutil.copy2(partial, f'{OUTPUT}/run_report.partial.json')

elapsed_min = (time.time() - started) / 60
print(f'Total elapsed time: {elapsed_min:.1f} minutes')
assert result.returncode == 0, \
    'Pipeline failed; inspect OUTPUT/logs and run_report.partial.json'

# Verify
verify = [
    sys.executable, '-m', 'sih_drone_pipeline', 'verify', '--output', OUTPUT,
]
assert subprocess.run(verify, cwd=PROJECT_DIR, env=env).returncode == 0, \
    'Artifact verification failed; see verification_report.json'

In [ ]:
# CELL 7 — inspect results and output file sizes
import json, os, glob

report = json.load(open(f'{OUTPUT}/run_report.json'))
verification = json.load(open(f'{OUTPUT}/verification_report.json'))

gpu_summary_path = f'{OUTPUT}/gpu_usage.summary.json'
gpu = json.load(open(gpu_summary_path)) if os.path.isfile(gpu_summary_path) else {}

print(json.dumps({
    'registered_images': report.get('sparse_metrics', {}).get('registered_images'),
    'mean_reprojection_error_px': report.get('sparse_metrics', {}).get('mean_reprojection_error_px'),
    'processing_minutes': round(report.get('wall_clock_seconds', 0) / 60, 2),
    'gps_validation': report.get('validation'),
    'targets': report.get('targets'),
    'artifact_checks_pass': verification.get('artifact_checks_pass'),
    'quality_checks': verification.get('quality_checks'),
    'production_ready': verification.get('production_ready'),
    'gpu_overall': gpu,
}, indent=2))

print('\nOutput files:')
for path in sorted(glob.glob(f'{OUTPUT}/*')):
    size = os.path.getsize(path) if os.path.isfile(path) else 0
    print(f'  {os.path.basename(path):30s} {size/1024/1024:9.1f} MB')

## Quality checks

- Most extracted frames should be registered.
- Mean reprojection error should normally be below 1 pixel.
- GPS alignment residual should be inspected (not independent proof of accuracy).
- This is a KABR wildlife dataset video — expect animal motion. `AI_MASK_DYNAMIC_OBJECTS=True` handles masking.
- Download `model.glb` and `viewer_metadata.json`, then open both in the metric web viewer.

### Next steps
- Re-run with `QUALITY = 'full'` and higher `TARGET_FRAMES` for production output.
- Add GCP control points for surveyed accuracy validation.